# 11a Natural Object Dataset (NOD) fMRI, converted

**Kaggle:** CPU, Internet on (OpenNeuro's public bucket). **Inputs:** none.

All 30 NOD people, ImageNet sessions: the released single-trial betas (59,412 fsLR-32k grayordinates) streamed one
run at a time and stored as float16 per person with each trial's image, and the ImageNet stimuli once each.
Structure only: no response statistic is computed, so the 21 target people stay unseen until the prespecified test
(P-8). NOD is CC0.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '11a-noddata', 'cpu-build'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"bucket": "https://s3.amazonaws.com/openneuro.org", "dataset": "ds004496", "n_vertices": 59412, "trials_per_run": 100, "image_size": 256, "jpeg_quality": 95, "subjects": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06", "sub-07", "sub-08", "sub-09", "sub-10", "sub-11", "sub-12", "sub-13", "sub-14", "sub-15", "sub-16", "sub-17", "sub-18", "sub-19", "sub-20", "sub-21", "sub-22", "sub-23", "sub-24", "sub-25", "sub-26", "sub-27", "sub-28", "sub-29", "sub-30"], "donors": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06", "sub-07", "sub-08", "sub-09"]}')

import numpy as np
import pandas as pd
from cross_subject_encoding.utils import config as C

In [ ]:
import re
import io
import subprocess
import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from PIL import Image
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel"], check=True)
import nibabel as nib

B, D = FIXED["bucket"], FIXED["dataset"]
NS = "{http://s3.amazonaws.com/doc/2006-03-01/}"
TMP = Path("/kaggle/tmp/nod")
TMP.mkdir(parents=True, exist_ok=True)
OUT = C.PROJECT_ROOT / "nod"
(OUT / "images").mkdir(parents=True, exist_ok=True)

def get(url, tries=5):
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=300) as r:
                return r.read()
        except Exception as e:
            if i == tries - 1:
                raise
            print("  retry", url.split("/")[-1], type(e).__name__)
            time.sleep(15 * (i + 1))

def listing(prefix):
    keys, token = [], None
    while True:
        q = {"list-type": "2", "prefix": prefix, "max-keys": "1000"}
        if token:
            q["continuation-token"] = token
        root = ET.fromstring(get(B + "?" + urllib.parse.urlencode(q)))
        keys += [c.find(NS + "Key").text for c in root.findall(NS + "Contents")]
        nxt = root.find(NS + "NextContinuationToken")
        if nxt is None:
            return keys
        token = nxt.text

stim_path = {}  # file name -> path under stimuli/ (events' stim_file, e.g. imagenet/n03598930/n03598930_18041.JPEG)
NAME = re.compile(r"n\d{8}_\d+\.JPEG")
report = {"subjects": {}}
saved = set()
for s in FIXED["subjects"]:
    t = time.monotonic()
    keys = listing(f"{D}/derivatives/ciftify/{s}/results/")
    runs = sorted({k.rsplit("/", 2)[-2] for k in keys if "task-imagenet" in k and k.endswith("_beta.dscalar.nii")},
                  key=lambda r: (int(re.search(r"ses-imagenet(\d+)", r).group(1)), int(re.search(r"run-(\d+)", r).group(1))))
    betas, names, mismatches = [], [], 0
    for run in runs:
        ses = re.search(r"(ses-imagenet\d+)", run).group(1)
        beta_file = TMP / "b.dscalar.nii"
        beta_file.write_bytes(get(f"{B}/{D}/derivatives/ciftify/{s}/results/{run}/{run}_beta.dscalar.nii"))
        b = np.asarray(nib.load(str(beta_file)).get_fdata(dtype=np.float32))
        if b.shape != (FIXED["trials_per_run"], FIXED["n_vertices"]):
            raise ValueError(f"{s} {run}: beta shape {b.shape}")
        run_no = int(re.search(r"run-(\d+)", run).group(1))  # derivatives use run-1, BIDS run-01
        ev = pd.read_csv(io.BytesIO(get(f"{B}/{D}/{s}/{ses}/func/{s}_{ses}_task-imagenet_run-{run_no:02d}_events.tsv")), sep="\t")
        run_names = [str(x).split("/")[-1] for x in ev["stim_file"]]
        stim_path.update({str(x).split("/")[-1]: str(x) for x in ev["stim_file"]})
        released = NAME.findall(get(f"{B}/{D}/derivatives/ciftify/{s}/results/{run}/{run}_label.txt").decode())
        if len(run_names) != FIXED["trials_per_run"]:
            raise ValueError(f"{s} {run}: {len(run_names)} events")
        mismatches += int(released != run_names)
        betas.append(b.astype(np.float16))
        names += run_names
    Y = np.concatenate(betas)
    if not np.isfinite(Y).all():
        raise ValueError(f"{s}: nonfinite betas")
    np.save(OUT / f"{s}_betas.npy", Y)
    (OUT / f"{s}_images.json").write_text(json.dumps(names))
    for n in names:
        if n in saved:
            continue
        img = Image.open(io.BytesIO(get(f"{B}/{D}/stimuli/{stim_path[n]}"))).convert("RGB").resize((FIXED["image_size"],) * 2, Image.BICUBIC)
        img.save(OUT / "images" / n.replace(".JPEG", ".jpg"), quality=FIXED["jpeg_quality"])
        saved.add(n)
    report["subjects"][s] = {"runs": len(runs), "trials": int(len(names)), "unique_images": len(set(names)),
                             "vertices": int(Y.shape[1]), "label_file_mismatching_runs": mismatches,
                             "role": "donor / development" if s in FIXED["donors"] else "target (fresh)"}
    print(s, report["subjects"][s], f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
all_names = [n for s in FIXED["subjects"] for n in json.loads((OUT / f"{s}_images.json").read_text())]
report["images_saved"] = len(saved)
report["images_seen_by_more_than_one_person"] = int(sum(v > 1 for v in pd.Series(all_names).value_counts()))
(C.METRICS_DIR / "11a_report.json").write_text(json.dumps(report, indent=2))
print("11A_REPORT_JSON " + json.dumps(report))